# **Quality Metrics Result Analysis**

In this notebook, the raw energy consumption data, with different configurations of inference batch sizes, is analyzed. The results are averaged over the five runs, along with the computation of the standard deviation for each tracked parameter.

In [99]:
# import required libraries
import pandas as pd
import os
import re

In [ ]:
# get working directory, necessary to gather the data to be analyzed
current_dir = os.getcwd()
print(f"Current Working Directory: {current_dir}")

## **Helper Functions**
Useful functions to properly format labels and to obtain final results from raw inference energy consumption data.

In [ ]:
def get_model_emissions(model_nomen, firstbaseline_csv, secondbaseline_csv, output_csv):
    
    """
    Aggregates GPU energy and emission data for a specific model from musiccaps
    and songdescriber datasets based on training steps.

    This function reads emission data from two CSV files, one for each baseline used,
    extracts the number of steps/model size from the 'project_name'
    column, groups the data by steps/size and baseline, sums the 'gpu_energy'
    and 'emissions' for each group, and combines the results
    into a single DataFrame. The combined data is then saved to a new CSV file.

    Parameters:
    -----------
    model_nomen : str
        The name of the model being analyzed. This name will be added as a
        column in the output DataFrame.

    firstbaseline_csv : str
        The file path to the CSV file containing emission data for the
        first dataset. This file is expected to have columns including
        'project_name', 'gpu_energy', and 'emissions'.

    secondbaseline_csv : str
        The file path to the CSV file containing emission data for the
        second dataset. This file is expected to have columns including
        'project_name', 'gpu_energy', and 'emissions'.

    output_csv : str
        The file path where the aggregated results will be saved as a CSV file.

    Returns
    -------
    pandas.DataFrame
        The aggregated DataFrame. Its columns depend on the model considered:
        - Diffusion model: ['model', 'steps', 'baseline', 'gpu_energy', 'emissions']
        - MusicGen/Magnet: ['model', 'baseline', 'gpu_energy', 'emissions']

    """

    if not model_nomen.startswith("MusicGen") and not model_nomen.startswith("Magnet"):

        def extract_steps(project_name):
            match = re.search(r'(\d+)-steps', project_name)
            return int(match.group(1)) if match else None

        df_first = pd.read_csv(firstbaseline_csv)
        df_second = pd.read_csv(secondbaseline_csv)

        df_first['steps'] = df_first['project_name'].apply(extract_steps)
        df_first['baseline'] = 'musiccaps'
        df_second['steps'] = df_second['project_name'].apply(extract_steps)
        df_second['baseline'] = 'songdescriber'

        agg_first = df_first.groupby(['steps', 'baseline']).agg({
            'gpu_energy': 'sum',
            'emissions': 'sum'
        }).reset_index()
        agg_second = df_second.groupby(['steps', 'baseline']).agg({
            'gpu_energy': 'sum',
            'emissions': 'sum'
        }).reset_index()

        agg_data = pd.concat([agg_first, agg_second], ignore_index=True)
        agg_data['model'] = model_nomen
        agg_data = agg_data[['model', 'steps', 'baseline', 'gpu_energy', 'emissions']]

        agg_data.to_csv(output_csv, index=False)
        print(f"Aggregated CSV for model '{model_nomen}' saved to '{output_csv}'.")
        
        return agg_data
    
    else:

        df_first = pd.read_csv(firstbaseline_csv)
        df_second = pd.read_csv(secondbaseline_csv)
        df_first['baseline'] = 'musiccaps'
        df_second['baseline'] = 'songdescriber'

        agg_first = df_first.groupby(['baseline']).agg({
            'gpu_energy': 'sum',
            'emissions': 'sum'
        }).reset_index()
        agg_second = df_second.groupby(['baseline']).agg({
            'gpu_energy': 'sum',
            'emissions': 'sum'
        }).reset_index()

        agg_data = pd.concat([agg_first, agg_second], ignore_index=True)
        agg_data['model'] = model_nomen
        agg_data = agg_data[['model', 'baseline', 'gpu_energy', 'emissions']]

        agg_data.to_csv(output_csv, index=False)
        print(f"Aggregated CSV for model '{model_nomen}' saved to '{output_csv}'.")
        return agg_data

In [102]:
def getdata(modello):
    """
    Retrieves and merges quality metrics and emission data for a specific model.

    This function reads FAD, CLAP, and Audiobox scores from CSV files located in a
    structured directory based on the model name. It then reads corresponding
    emission data (emissions, gpu_energy) and merges it with the quality
    metrics based on 'steps' and 'baseline'. The merged data, excluding
    the first column, is printed to the console.

    Parameters:
    -----------
    modello : str
        The name of the model for which to retrieve and merge data. This is used
        to construct the file paths for quality metrics and emission data.

    Returns:
    --------
    None
        The function does not return a value but prints the merged DataFrame
        (excluding the first column and without the index) to the standard output.

    Output:
    -------
    - The function prints a string representation of the merged pandas DataFrame
      (excluding the first column and the index) to the console.
    """
    merged_data = pd.DataFrame()

    suffix = re.compile(r'(Small|Medium|Large)$', re.IGNORECASE)
    m = suffix.search(modello)

    for metr in ["FAD", "CLAP"]:
        file_path = rf"{current_dir}\results\quality_metrics\{metr}\{modello}_{metrics[metr][0]}s.csv"
        data = pd.read_csv(file_path)
        if m:
            base = modello[:m.start()]
            en_path = rf"{current_dir}\results\quality_metrics\Emissions\{base}\{modello}_emissions.csv"
        else:
            en_path = rf"{current_dir}\results\quality_metrics\Emissions\{modello}\{modello}_emissions.csv"
        energy = pd.read_csv(en_path)
        for col in ['emissions', 'gpu_energy']:
            if m:
                data = data.merge(energy[['baseline', col]], on=['baseline'], how='left')                
            else:
                data = data.merge(energy[['steps', 'baseline', col]], on=['steps', 'baseline'], how='left')

        if merged_data.empty: merged_data = data
        else: merged_data.insert(loc=5, column="clap_score", value=data["clap_score"])

    # Audiobox
    if m:
        base = modello[:m.start()]
        file_path = rf"{current_dir}\results\quality_metrics\Audiobox\{base}\{modello}_audiobox.csv"
        en_path = rf"{current_dir}\results\quality_metrics\Emissions\{base}\{modello}_emissions.csv"
    else:
        file_path = rf"{current_dir}\results\quality_metrics\Audiobox\{modello}\{modello}_audiobox.csv"
        en_path = rf"{current_dir}\results\quality_metrics\Emissions\{modello}\{modello}_emissions.csv"
    data = pd.read_csv(file_path)
    energy = pd.read_csv(en_path)
    if 'Steps' not in energy.columns and 'steps' in energy.columns:
        energy = energy.rename(columns={'steps': 'Steps'})
    if 'Baseline' not in energy.columns and 'baseline' in energy.columns:
        energy = energy.rename(columns={'baseline': 'Baseline'})
    for col in ['emissions', 'gpu_energy']:
        if m:
            data = data.merge(energy[['Baseline', col]], on=['Baseline'], how='left')
        else:
            data = data.merge(energy[['Steps', 'Baseline', col]], on=['Steps', 'Baseline'], how='left')

    if merged_data.empty: merged_data = data
    else: 
        merged_data.insert(loc=6, column="Avg_CE", value=data["Avg_CE"])
        merged_data.insert(loc=7, column="Avg_CU", value=data["Avg_CU"])
        merged_data.insert(loc=8, column="Avg_PC", value=data["Avg_PC"])
        merged_data.insert(loc=9, column="Avg_PQ", value=data["Avg_PQ"])

    print(merged_data.iloc[:, 1:].to_string(index=False))

In [103]:
def print_metrics(modello):
    print("-" * 70)
    print(modello.center(70))
    print("-" * 70)
    getdata(modello)
    print("\n")

## **Data analysis**
Using the previously defined functions, we now can analyze the data obtained for the five different runs for all considered models.

In [ ]:
metrics = {"FAD": ["fad_score", "FAD"],
           "CLAP": ["clap_score", "CLAP Score"],
           "Audiobox": ["audiobox", "Audiobox"]}
model_list = ["musicAudioLDM", "musicAudioLDM2", "MusicLDM", "musicSAO", "musicTango", "ACEStep", "AudioX",
               "MusicGenSmall", "MusicGenMedium", "MusicGenLarge", "MagnetSmall", "MagnetMedium"]

suffix = re.compile(r'(Small|Medium|Large)$', re.IGNORECASE)

for model in model_list:
    m = suffix.search(model)
    if m:
        base = model[:m.start()]
        musiccaps_file = fr"{current_dir}\results\quality_metrics\Emissions\{base}\{model}_musiccaps.csv"
        songdescriber_file = fr"{current_dir}\results\quality_metrics\Emissions\{base}\{model}_songdescriber.csv"
        output_file = fr"{current_dir}\results\quality_metrics\Emissions\{base}\{model}_emissions.csv"
        get_model_emissions(model, musiccaps_file, songdescriber_file, output_file)
    else:
        musiccaps_file = fr"{current_dir}\results\quality_metrics\Emissions\{model}\{model}_musiccaps.csv"
        songdescriber_file = fr"{current_dir}\results\quality_metrics\Emissions\{model}\{model}_songdescriber.csv"
        output_file = fr"{current_dir}\results\quality_metrics\Emissions\{model}\{model}_emissions.csv"
        get_model_emissions(model, musiccaps_file, songdescriber_file, output_file)

Aggregated CSV for model 'musicAudioLDM' saved to 'c:\Users\ricca\Documents\GitHub\musicdiff-consumption\results\quality_metrics\Emissions\musicAudioLDM\musicAudioLDM_emissions.csv'.
Aggregated CSV for model 'musicAudioLDM2' saved to 'c:\Users\ricca\Documents\GitHub\musicdiff-consumption\results\quality_metrics\Emissions\musicAudioLDM2\musicAudioLDM2_emissions.csv'.
Aggregated CSV for model 'MusicLDM' saved to 'c:\Users\ricca\Documents\GitHub\musicdiff-consumption\results\quality_metrics\Emissions\MusicLDM\MusicLDM_emissions.csv'.
Aggregated CSV for model 'musicSAO' saved to 'c:\Users\ricca\Documents\GitHub\musicdiff-consumption\results\quality_metrics\Emissions\musicSAO\musicSAO_emissions.csv'.
Aggregated CSV for model 'musicTango' saved to 'c:\Users\ricca\Documents\GitHub\musicdiff-consumption\results\quality_metrics\Emissions\musicTango\musicTango_emissions.csv'.
Aggregated CSV for model 'ACEStep' saved to 'c:\Users\ricca\Documents\GitHub\musicdiff-consumption\results\quality_metric

In [108]:
# analyze single models
model_name = "musicSAO" # model = "musicAudioLDM2" # model = "MusicGenSmall" # . . .

print_metrics(model_name)

----------------------------------------------------------------------
                               musicSAO                               
----------------------------------------------------------------------
 steps      baseline  fad_score_laion  fad_score_encodec  clap_score   Avg_CE   Avg_CU   Avg_PC   Avg_PQ  emissions  gpu_energy
    10     musiccaps         0.412123          59.390971    0.243080 5.452521 6.925653 3.345232 7.115593   0.009696    0.022719
    25     musiccaps         0.380551          42.582902    0.286439 6.002226 7.172918 3.280720 7.304504   0.022402    0.053121
    50     musiccaps         0.378061          44.461566    0.294623 6.023632 7.179566 3.448857 7.336824   0.043571    0.103731
   100     musiccaps         0.371964          44.032996    0.298530 6.123726 7.240622 3.461872 7.388155   0.085881    0.204981
   150     musiccaps         0.368183          43.383964    0.298713 6.129312 7.235404 3.464309 7.396066   0.128313    0.306499
   200     musiccap

In [106]:
for model in model_list:
    print_metrics(model)

----------------------------------------------------------------------
                            musicAudioLDM                             
----------------------------------------------------------------------
 steps      baseline  fad_score_laion  fad_score_encodec  clap_score   Avg_CE   Avg_CU   Avg_PC   Avg_PQ  emissions  gpu_energy
    10     musiccaps         0.464203          45.605617    0.315187 5.614217 6.421329 5.011989 6.453657   0.002723    0.005074
    25     musiccaps         0.388265          39.425721    0.341409 5.759664 6.533918 5.143729 6.603574   0.005012    0.009456
    50     musiccaps         0.380096          37.782889    0.348740 5.807556 6.533530 5.237712 6.643159   0.008884    0.016819
   100     musiccaps         0.368473          36.893895    0.351775 5.771050 6.490754 5.262980 6.617494   0.016325    0.031184
   150     musiccaps         0.373279          38.062658    0.352357 5.750705 6.517587 5.357030 6.644884   0.024018    0.045774
   200     musiccap